# 03 — Extracteur de labels NLP, validation sur les 58 gold

L'extracteur `src/nlp/` (lexique × 9 langues, négation directionnelle — droite en turc)
produit pour chaque étude 12 labels `(0/1/None)` + 12 confiances.
`None` = pas de signal → la cible sera **masquée** dans la loss vision.

Validation : les 58 gold servent **d'étalon** (AUC/F1 par cible). Le vocabulaire n'est
pas retuné dessus ; les corrections viennent des sanity checks et des formulations
fréquentes des rapports **non-gold**.

Résultat v1 : macro-AUC ≈ 0.76 (les cibles non couvertes sont imputées à 0.5, borne
pessimiste), Baker ≈ 0.83, Effusion ≈ 0.68 — cohérent avec le floor publié.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import pandas as pd

from src.nlp.extractor import extract_text
from src.nlp.lang_detect import detect_lang
from src.nlp.lexicon import TARGETS

train = pd.read_csv(Path.cwd().parent / 'data/csv/train.csv')
labeled = train[train[TARGETS].notna().all(axis=1)].copy()
labeled['lang'] = labeled['Report'].map(detect_lang)
print(f'{len(labeled)} études gold | langues:', labeled.lang.value_counts().to_dict())

In [ ]:
# Extraction sur les 58 gold (CPU, ~10 s)
rows = []
for _, r in labeled.iterrows():
    pred = extract_text(str(r['Report']), r['lang'])
    row = {'StudyInstanceUID': r['StudyInstanceUID'], 'lang': r['lang']}
    for t in TARGETS:
        row[t] = pred[t][0]
        row[t + '_conf'] = pred[t][1]
    rows.append(row)
preds = pd.DataFrame(rows)
preds.head()

In [ ]:
from sklearn.metrics import f1_score, roc_auc_score

stats = []
for t in TARGETS:
    y_true = labeled[t].astype(int).values
    yp = preds[t].values
    mask = np.array([v is not None and not (isinstance(v, float) and np.isnan(v)) for v in yp])
    conf = preds[t + '_conf'].values
    score = np.where(mask, np.where(pd.Series(yp).fillna(-1).astype(float).values == 1, conf, 1 - conf), 0.5)
    auc = roc_auc_score(y_true, score) if len(np.unique(y_true)) > 1 else np.nan
    yp_bin = np.where(mask, pd.Series(yp).fillna(0).astype(float).values, 0)
    stats.append({'cible': t, 'AUC': auc, 'F1': f1_score(y_true, yp_bin, zero_division=0),
                  'couverture': mask.mean(), 'prev_gold': y_true.mean(), 'prev_pred': yp_bin.mean()})

stats = pd.DataFrame(stats)
display(stats.round(3))
print(f"macro-AUC : {stats.AUC.mean():.3f} | macro-F1 : {stats.F1.mean():.3f}")

In [ ]:
# Erreurs par cible et par langue (debug, pas de tuning de seuils)
for t in ['ACL', 'Effusion', 'Medial Meniscus']:
    yp = preds[t]
    mask = yp.notna()
    err = labeled.loc[mask.values, t].astype(int).values != yp[mask].astype(float).values
    sub = preds.loc[mask][err].merge(labeled[['StudyInstanceUID']], on='StudyInstanceUID')
    print(f'{t}: {err.sum()} erreurs / {mask.sum()} couvertes')
    if len(sub):
        display(sub.head(8))

## Lecture

- Couverture = fraction des études où l'extracteur a un signal (le reste sera masqué en loss).
- Effusion a la meilleure couverture (0.84) mais un AUC modeste : label bruité, seuil de
  confiance plus strict en phase 2.
- OA × 3 et Synovitis : couverture faible (~0.25) → beaucoup de masquage, c'est voulu
  (conservateur > bruité).
- Prochaine étape : `04_pseudo_labels_split.ipynb` — extraction sur les 4 349 rapports
  non-gold, masquage par confiance, split étude stratifié.